In [1]:
!pip install -q --upgrade diffusers transformers accelerate
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q scipy
import diffusers
print(diffusers.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 93.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
0.39.0.dev0


In [2]:
import torch
from diffusers import LTXPipeline
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# メモリ解放
torch.cuda.empty_cache()

pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16
)

# VRAM節約の設定
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()  # これも追加

prompt = """
anime style, japanese animation, cel shading,
anime girl walking in a forest, studio ghibli style,
correct anatomy, correct proportions,
full body, detailed face,
stable character design, consistent clothing,
vibrant colors, clean lines, smooth motion,
high quality anime
"""

negative_prompt = """
realistic, 3D, photorealistic, low quality, blurry,
distorted, deformed, warped, morphing, glitchy,
bad anatomy, extra limbs, melting, disfigured,
changing clothes, inconsistent design,
wrong proportions, short legs, elongated torso
"""

video = pipe(
    #image=image,
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=49,
    width=576,   # 512→576（32の倍数）
    height=320,  # そのままOK
    num_inference_steps=50,
    guidance_scale=9.5,
).frames[0]

from diffusers.utils import export_to_video
export_to_video(video, "output.mp4", fps=24)
print("生成完了！")


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

生成完了！


In [3]:
import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# I2Vパイプラインロード
pipe_i2v = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16
)
pipe_i2v.enable_model_cpu_offload()
pipe_i2v.vae.enable_tiling()
pipe_i2v.vae.enable_slicing()

# 画像読み込み
image = load_image("your_image.png")  # パスを変更

prompt = """
anime style, japanese animation, cel shading,
sharp outlines, crisp edges, clean lineart,
smooth motion, stable character design,
studio ghibli style, high quality anime
"""

negative_prompt = """
realistic, 3D, photorealistic, low quality,
blurry edges, distorted, deformed, bad anatomy
"""

video = pipe_i2v(
    image=image,
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=49,
    width=576,
    height=320,
    num_inference_steps=50,
    guidance_scale=9.5,
).frames[0]

export_to_video(video, "output_i2v.mp4", fps=24)
print("I2V完了！")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ValueError: Incorrect path or URL. URLs must start with `http://` or `https://`, and your_image.png is not a valid path.